# Stage 3, step 3: build BioLangFusion and check it against the published numbers

Step 1 pinned down the four benchmarks and recovered the preprocessing rule that reproduces
the paper's row counts. Step 2 wrote token-level embeddings for the three encoders
BioLangFusion fuses. This notebook builds the model, the codon grid, the three fusion heads,
and the shared TextCNN prediction head, and asks one question: **does it reach the numbers
the paper reports?**

Nothing here is new research, and that is the point. Every later result in this project, the
Coupled Mamba head, cross-attention as a fourth head, the encoder swaps, the alignment
ablation, is a comparison against this baseline. A baseline that has not been checked against
a published number is not a baseline, it is an assumption. If the reproduction fails we find
out now, with two notebooks of work behind us, rather than in January with six.

The model is not written in this notebook. It lives in `../shared_code/fusion/`, so that
Stage 4 imports the same code rather than a copy of it:

| file | what it holds |
| --- | --- |
| `codon_grid.py` | resampling the three streams onto the protein frame |
| `heads.py` | the three fusion strategies |
| `textcnn.py` | the shared prediction head |
| `blf.py` | the cached streams, batching, and the assembled model |
| `train.py` | the training loop, the early stop, and the scoring |

Each file's docstring records the decisions the paper leaves open, and
`tests/test_fusion_model.py` checks the shapes and the masking invariances on small tensors
without touching a checkpoint or a cache.

---

In [1]:
# Standard library
import os
import time
from pathlib import Path

# Run any operation the Apple GPU backend lacks on the CPU; must precede importing torch
os.environ.setdefault("PYTORCH_ENABLE_MPS_FALLBACK", "1")

# Data handling
import numpy as np
import pandas as pd
import torch

# Shared project code in Code/shared_code; this notebook lives in Code/stage3
import sys
CODE_DIR = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "shared_code").is_dir()), Path.cwd() / "Code")
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))
from shared_code.fusion import blf, codon_grid, heads, textcnn, train

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 40)
print("torch", torch.__version__, "| Apple GPU available:", torch.backends.mps.is_available())

torch 2.12.1 | Apple GPU available: True


## The target

BioLangFusion's Table 1 is the whole comparison: seven encodings, the three single-modality
baselines it fuses among them, and the three fusion heads, on five benchmarks. Table 4 adds
two ablations of the gated head. Those are the numbers this notebook has to hit.

In [2]:
# BioLangFusion Table 1 in full, as printed. Spearman correlation everywhere except E. coli,
# which is accuracy on a 0 to 100 scale. The first four rows are encoders the paper compares
# against but does not fuse; rows five to seven are the three it fuses.
TABLE1 = pd.DataFrame(
    [
        ("Evo",                     "other",    0.653, 0.579, 42.556, 0.403, 0.360),
        ("SpliceBERT",              "other",    0.802, 0.778, 48.455, 0.522, 0.718),
        ("ESM-2 (650M)",            "other",    0.825, 0.734, 46.348, 0.536, 0.679),
        ("ESM-2 (3B)",              "other",    0.772, 0.721, 46.208, 0.537, 0.700),
        ("ESM-2 (8M)",              "single",   0.806, 0.695, 49.017, 0.539, 0.711),
        ("RNA-FM",                  "single",   0.841, 0.767, 52.949, 0.553, 0.743),
        ("Nucleotide Transformer",  "single",   0.780, 0.804, 41.292, 0.530, 0.732),
        ("Concatenation",           "fusion",   0.831, 0.805, 50.280, 0.539, 0.764),
        ("MIL + Entropy",           "fusion",   0.864, 0.824, 52.107, 0.563, 0.760),
        ("Cross Attention",         "fusion",   0.828, 0.812, 53.932, 0.550, 0.765),
    ],
    columns=["encoding", "kind", "cov_vac_degradation", "fungal_expression",
             "ecoli_proteins", "mrna_stability", "ab1"],
).set_index("encoding")

# Table 4, the two ablations of the gated head, on the same five benchmarks.
TABLE4 = pd.DataFrame(
    [
        ("MIL without entropy",                     0.854, 0.826, 51.960, 0.556, 0.753),
        ("MIL with shared projection (+ entropy)",  0.859, 0.813, 50.140, 0.563, 0.755),
        ("MIL + entropy",                           0.864, 0.824, 52.107, 0.563, 0.760),
    ],
    columns=["ablation", "cov_vac_degradation", "fungal_expression",
             "ecoli_proteins", "mrna_stability", "ab1"],
).set_index("ablation")

display(TABLE1)
display(TABLE4)

,kind,cov_vac_degradation,fungal_expression,ecoli_proteins,mrna_stability,ab1
encoding,,,,,,
Evo,other,0.653,0.579,42.556,0.403,0.360
SpliceBERT,other,0.802,0.778,48.455,0.522,0.718
ESM-2 (650M),other,0.825,0.734,46.348,0.536,0.679
ESM-2 (3B),other,0.772,0.721,46.208,0.537,0.700
ESM-2 (8M),single,0.806,0.695,49.017,0.539,0.711
RNA-FM,single,0.841,0.767,52.949,0.553,0.743
Nucleotide Transformer,single,0.780,0.804,41.292,0.530,0.732
Concatenation,fusion,0.831,0.805,50.280,0.539,0.764
MIL + Entropy,fusion,0.864,0.824,52.107,0.563,0.760


,cov_vac_degradation,fungal_expression,ecoli_proteins,mrna_stability,ab1
ablation,,,,,
MIL without entropy,0.854,0.826,51.960,0.556,0.753
MIL with shared projection (+ entropy),0.859,0.813,50.140,0.563,0.755
MIL + entropy,0.864,0.824,52.107,0.563,0.760


Two things in those tables are worth reading before trying to match them.

**On fungal expression the fusion gain is small and the ordering is the claim.** The best
single encoder, the Nucleotide Transformer at 0.804, is one thousandth below plain
concatenation at 0.805. The paper's headline, that fusion beats the best unimodal baseline,
rests on the gated head's 0.824, a gain of 0.020. A reproduction that lands every variant
within a few hundredths but puts them in the wrong order has not reproduced the result. The
ordering is checked explicitly below, separately from the absolute values.

**Table 4 contradicts the sentence that introduces it.** The text says "entropy regularizer
improves prediction performance", and on three of the five benchmarks it does. On fungal
expression the gated head *without* entropy scores 0.826 against 0.824 with it, and on mRNA
stability the two are identical at 0.563. So on the benchmark this notebook checks against,
the paper's own ablation does not support its claim. Both settings are run here, which costs
one extra training run and turns a contradiction in the paper into a measurement.

## Why this checks against fungal expression

Four benchmarks are in hand. Fungal expression is the one to check a build against, and the
reason is in what step 1 measured rather than in convenience.

**mRNA stability, the largest, has a contaminated test split.** Step 1 found 23,929 rows over
11,303 distinct sequences, and 2,444 of the 3,579 published test rows, 68 percent, reuse a
sequence that also appears in training. Only 985 of the 3,021 distinct test sequences are
unseen. Worse, 4,549 of the 4,579 repeated sequences carry *differing* labels, with a median
spread of 0.94 on a range of about 6, so a model that memorizes a training sequence is scored
against a different number for the same input. Published Spearman values of 0.553 and 0.563
on that split are not a target a correct implementation should want to hit: matching them
would be evidence about leakage, not about the build.

**CoV-Vac is the cheapest but leans on our own decision.** Its sequences are 81-nucleotide
fragments cut from a construct rather than coding sequences, and step 2 had to choose how to
feed them to a protein encoder. Only 0.7 percent translate to a full reading frame under the
stop-at-first-stop rule, so `grid_protein` translates every codon and writes stops as X. That
decision is ours, not the paper's, and a CoV-Vac score measures it as much as the build. It
is the right second check, not the first.

**E. coli proteins is classification**, scored by accuracy, so it exercises a different loss
and a different metric. Useful as a third check that the loss switch works.

**Fungal expression has clean published splits, 3,138 rows, regression, and published values
for all six variants.** It is the one benchmark where matching the numbers means the build is
right.

## What the paper leaves open

Five decisions are not determined by the paper, and a reproduction has to make them. Each is
recorded in the docstring of the module that makes it, so the reasoning sits next to the code
rather than only here.

1. **The transposed convolution is specified twice, differently.** Equation 1 gives kernel 2,
   stride 2, which doubles the DNA stream exactly. Appendix List 1 and List 2 give kernel 3,
   stride 2, padding 2, which returns `2n - 3`, three positions short of doubling. Both are
   implemented; the appendix setting is the default, because the appendix is what the paper
   calls the detailed architecture. Neither lands on `T'` exactly, so the stream is fitted to
   the protein frame afterwards, which is the paper's own line "Padding: sequence length
   aligned to protein". The two settings turn out not to be equivalent, and the section after
   the verdict measures the difference; read the sweep's result before concluding anything
   from it.

2. **The prediction head is called fixed and given two widths.** The text says all fusion
   methods "share the same fixed TextCNN prediction head to ensure a fair comparison", then
   List 1 specifies 1,280 Conv1D output channels for the gated variant and List 2 specifies
   100 for the cross-attention variant, a factor of nearly thirteen in head capacity. One
   width for every variant is what a fair comparison means, so the main run uses 100, the
   narrower of the two, and List 1's 1,280 is run once at the end to see whether head width,
   rather than fusion, explains any gap.

3. **The DNA embedding width in Table 3 is the wrong number.** Table 3 gives the Nucleotide
   Transformer an embedding dimension of 4,107, and Section 2.2.1 introduces a projection
   "since d_DNA is significantly larger than d_RNA and d_Prot". 4,107 is that checkpoint's
   vocabulary size; its hidden states are 512 wide, which step 2's caches confirm. The DNA
   stream is in fact the narrowest of the three and the imbalance the projection was
   introduced to fix does not exist. The projection is kept, because it is in the
   architecture, at 600, the only projection width the paper names.

4. **Nothing says what the early stop and the scheduler watch.** Both watch the validation
   score, the same quantity the paper reports, and the weights of the best validation epoch
   are restored before the test split is touched.

5. **Two means would otherwise depend on batching.** The gated head summarizes each modality
   by a mean over codons, and rows are padded to the longest in their batch, so an unweighted
   mean would make a sequence's representation depend on which sequences it traveled with.
   The mask is applied. The same applies to the upsampled DNA stream, which is trimmed to each
   row's own convolution output length before the frame mask.

Point 5 is not a detail. A model whose prediction for a sequence changes with its batch mates
produces results that cannot be reproduced, including by us. The check below measures it.

In [3]:
# Structural checks, on random tensors and on the CPU: no cache, no checkpoint, no training.
# These answer "is the architecture the one in the paper", separately from "does it score".
torch.manual_seed(0)
W = {"dna": 512, "rna": 640, "prot": 320}      # the widths step 2's caches actually hold
checks = []


def note(name, passed, detail):
    checks.append({"check": name, "result": "ok" if passed else "PROBLEM", "detail": detail})


# 1. Every stream lands on the protein frame, under both published convolution settings.
for setting in codon_grid.TCONV_SETTINGS:
    grid = codon_grid.CodonGrid(W["dna"], tconv=setting).eval()
    nt = 999
    dna = torch.randn(2, nt // 6 + nt % 6, W["dna"])
    rna = torch.randn(2, nt, W["rna"])
    prot = torch.randn(2, nt // 3, W["prot"])
    codons = torch.tensor([nt // 3] * 2)
    tokens = torch.tensor([nt // 6 + nt % 6] * 2)
    with torch.no_grad():
        streams, mask = grid(dna, rna, prot, codons, tokens)
    shapes = {m: s.shape[1] for m, s in streams.items()}
    note(f"codon frame, tconv={setting}", set(shapes.values()) == {nt // 3},
         f"999 nt -> {shapes}, expected {nt // 3} everywhere")

# 2. A row is resampled the same alone as it is beside a longer row.
grid = codon_grid.CodonGrid(W["dna"]).eval()
short, long = 300, 999
dna = torch.zeros(2, long // 6 + long % 6, W["dna"])
rna = torch.zeros(2, long, W["rna"])
prot = torch.zeros(2, long // 3, W["prot"])
for row, n in enumerate((short, long)):
    dna[row, :n // 6 + n % 6] = torch.randn(n // 6 + n % 6, W["dna"])
    rna[row, :n] = torch.randn(n, W["rna"])
    prot[row, :n // 3] = torch.randn(n // 3, W["prot"])
codons = torch.tensor([short // 3, long // 3])
tokens = torch.tensor([short // 6 + short % 6, long // 6 + long % 6])
with torch.no_grad():
    batched, _ = grid(dna, rna, prot, codons, tokens)
    alone, _ = grid(dna[:1, :short // 6 + short % 6], rna[:1, :short], prot[:1, :short // 3],
                    codons[:1], tokens[:1])
drift = max(float((batched[m][:1, :short // 3] - alone[m]).abs().max()) for m in alone)
note("a row is unchanged by its batch mates", drift == 0.0, f"largest difference {drift:.3g}")

# 3. The prediction head ignores padding.
head = textcnn.TextCNN(W["rna"], 1, channels=16).eval()
real = torch.randn(1, 120, W["rna"])
padded = torch.cat([real, torch.randn(1, 60, W["rna"])], dim=1)
padded_mask = torch.zeros(1, 180, dtype=torch.bool)
padded_mask[:, :120] = True
with torch.no_grad():
    gap = float((head(real, torch.ones(1, 120, dtype=torch.bool)) - head(padded, padded_mask)).abs().max())
note("the head ignores padding", gap < 1e-5, f"largest difference {gap:.3g}")

# 4. The gated head gives one weight per modality per sequence, summing to one.
streams, mask = grid(dna, rna, prot, codons, tokens)
_, aux = heads.GatedMILFusion(W)(streams, mask)
alpha = aux["alpha"]
note("gated attention: one weight per modality per sequence",
     alpha.shape == (2, 3) and bool(torch.allclose(alpha.sum(1), torch.ones(2))),
     f"shape {tuple(alpha.shape)}, row sums {[round(float(v), 6) for v in alpha.detach().sum(1)]}")

# 5. Every parameter of every head receives a finite gradient.
for name, factory in heads.FUSIONS.items():
    model = factory(W)
    fused, _ = model({m: s.detach() for m, s in streams.items()}, mask)
    fused.sum().backward()
    missing = [n for n, p in model.named_parameters()
               if p.requires_grad and (p.grad is None or not torch.isfinite(p.grad).all())]
    note(f"gradients reach every parameter, {name}", not missing,
         "all finite" if not missing else f"missing: {missing}")

pd.DataFrame(checks).set_index("check")

,result,detail
check,,
"codon frame, tconv=appendix",ok,"999 nt -> {'dna': 333, 'rna': 333, 'prot': 333..."
"codon frame, tconv=equation",ok,"999 nt -> {'dna': 333, 'rna': 333, 'prot': 333..."
a row is unchanged by its batch mates,ok,largest difference 0
the head ignores padding,ok,largest difference 0
gated attention: one weight per modality per sequence,ok,"shape (2, 3), row sums [1.0, 1.0]"
"gradients reach every parameter, concat",ok,all finite
"gradients reach every parameter, mil",ok,all finite
"gradients reach every parameter, cross",ok,all finite


---

### Settings

`RESULTS_DIR` is what makes this notebook restartable. Every training run writes a JSON
result there when it finishes and a checkpoint every ten epochs while it is going, so
rerunning the long cell after an interruption picks up where it stopped rather than starting
the sweep again. Delete a result file to force that run again.

In [4]:
BENCHMARK = "fungal_expression"          # clean splits, published values for all six variants
CACHE_ROOT = Path("embeddings")          # written by 02_token_embeddings.ipynb
RESULTS_DIR = Path("results") / BENCHMARK

CHANNELS = 100                           # TextCNN output channels; List 2's width, see above
BATCH_SIZE = 32                          # List 3
MAX_EPOCHS = 500                         # List 3
LAMBDAS = (0.01, 0.5, 1.0)               # the paper selects from this set on validation

DEVICE = ("mps" if torch.backends.mps.is_available()
          else "cuda" if torch.cuda.is_available() else "cpu")

# Cheapest first, so an interrupted night still leaves the baselines and the gated head done.
# The gated head's lambda sweep is the paper's: "selected from the set {0.01, 0.5, 1} via
# validation performance". Lambda 0 is Table 4's "MIL without entropy" ablation.
RUNS = ([("unimodal:prot", {}), ("unimodal:dna", {}), ("unimodal:rna", {}), ("concat", {})]
        + [("mil", {"entropy_lambda": lam}) for lam in LAMBDAS]
        + [("mil", {"entropy_lambda": 0.0}), ("cross", {})])

RESULTS_DIR.mkdir(parents=True, exist_ok=True)
STREAMS = blf.BenchmarkStreams.open(BENCHMARK, CACHE_ROOT)
print(f"{BENCHMARK}: {len(STREAMS.rows):,} rows, {STREAMS.task}, "
      f"widths {STREAMS.widths}, device {DEVICE}")
print("splits:", {s: len(STREAMS.split_rows(s)) for s in ("train", "val", "test")})
print(f"{len(RUNS)} runs:", ", ".join(
    v + (f" (lam {o['entropy_lambda']})" if o else "") for v, o in RUNS))

fungal_expression: 3,138 rows, regression, widths {'dna': 512, 'rna': 640, 'prot': 320}, device mps
splits: {'train': 2153, 'val': 555, 'test': 430}
9 runs: unimodal:prot, unimodal:dna, unimodal:rna, concat, mil (lam 0.01), mil (lam 0.5), mil (lam 1.0), mil (lam 0.0), cross


### What this will cost

`BenchmarkStreams.open` has already done the part that catches a broken cache: it checks that
every cache holds one row per benchmark sequence, that the RNA stream is one token per
nucleotide, and that the protein stream is one token per codon. If that passed, the streams
line up on the codon grid.

The cost itself is dominated by one variant. Cross-attention builds a joint context of three
times the codon frame, up to 999 positions here, and attends to it once per modality, so it
runs roughly seven times slower per epoch than the gated head. Measured on four CPU threads
while this notebook was written, per epoch on this benchmark: the two short unimodal
baselines 0.09 minutes, the gated head 0.52, RNA-FM 0.53, concatenation 0.59, and
cross-attention 3.81. The cell below measures the same thing on whatever device this is
actually running on, which for an Apple GPU should be substantially faster.

How many epochs each run needs is the part no measurement settles in advance. A 20-epoch
probe of the DNA baseline reached test Spearman 0.794 against the published 0.804 with its
validation score still climbing, so these runs converge in the right direction but are not
close to finished at 20. The projection below is given for 100 and 250 epochs to bracket it.

In [5]:
# Three batches of each variant on this device, forward and backward, after a warm-up batch.
# Cross-attention is the one that can run out of memory at batch size 32, so a failure is
# caught and reported here rather than being allowed to stop the notebook.
train_batches = -(-len(STREAMS.split_rows("train")) // BATCH_SIZE)


def target_for(batch):
    return batch.labels if STREAMS.task == "classification" else batch.labels.view(-1, 1).float()


def time_variant(variant):
    """Minutes per epoch for one variant, or NaN and the reason it could not be measured."""
    model = blf.build(STREAMS, variant, channels=CHANNELS).to(DEVICE)
    params = sum(p.numel() for p in model.parameters())
    optimiser = torch.optim.Adam(model.parameters(), lr=3e-5)
    criterion = train.loss_function(STREAMS.task)
    batches = iter(blf.loader(STREAMS, "train", batch_size=BATCH_SIZE, shuffle=False))
    try:
        warm = next(batches).to(DEVICE)
        out, _ = model(warm)
        criterion(out, target_for(warm)).backward()
        started, measured = time.perf_counter(), 0
        for batch in batches:
            batch = batch.to(DEVICE)
            optimiser.zero_grad()
            out, _ = model(batch)
            criterion(out, target_for(batch)).backward()
            optimiser.step()
            measured += 1
            if measured == 3:
                break
        per_epoch, note = (time.perf_counter() - started) / measured * train_batches / 60, ""
    except (RuntimeError, MemoryError) as problem:
        per_epoch, note = float("nan"), f"{type(problem).__name__}: {str(problem)[:90]}"
    finally:
        del model, optimiser
        if DEVICE == "mps":
            torch.mps.empty_cache()
    return {"params": params, "min/epoch": round(per_epoch, 2), "note": note}


# Each model once, not once per lambda: the sweep reuses the same architecture.
timing = pd.DataFrame([{"variant": v, **time_variant(v)}
                       for v in dict.fromkeys(v for v, _ in RUNS)]).set_index("variant")
timing["hours @100"] = (timing["min/epoch"] * 100 / 60).round(1)
timing["hours @250"] = (timing["min/epoch"] * 250 / 60).round(1)

runs_per_variant = pd.Series([v for v, _ in RUNS]).value_counts()
total = (timing["min/epoch"] * timing.index.map(runs_per_variant)).sum()
display(timing)
print(f"whole sweep, {len(RUNS)} runs: {total * 100 / 60:.1f} hours at 100 epochs each, "
      f"{total * 250 / 60:.1f} hours at 250")
for variant, note in timing.loc[timing["note"] != "", "note"].items():
    print(f"NOT MEASURED  {variant}: {note}")


,params,min/epoch,note,hours @100,hours @250
variant,,,,,
unimodal:prot,384601,0.16,,0.3,0.7
unimodal:dna,615001,0.07,,0.1,0.3
unimodal:rna,768601,0.07,,0.1,0.3
concat,2751945,0.14,,0.2,0.6
mil,2753246,0.07,,0.1,0.3
cross,7805145,0.51,,0.8,2.1


whole sweep, 9 runs: 2.0 hours at 100 epochs each, 5.1 hours at 250


Every variant should appear with a finite time. A failure here, rather than four hours in, is
the point of the cell: cross-attention is the one that can run out of memory at batch size 32,
because the joint context is three times the codon frame. If it does, the escape hatch is a
smaller batch for that variant alone, which is a deviation from List 3 and has to be recorded
as one.

### A short run before the long one

In [6]:
# Five epochs of the cheapest variant. The point is not the score, it is that the loss falls
# and the validation score moves on this device before a night is committed to it.
probe = train.fit(blf.build(STREAMS, "unimodal:prot", channels=CHANNELS), STREAMS,
                  train.TrainConfig(max_epochs=5, batch_size=BATCH_SIZE, channels=CHANNELS,
                                    early_stopping_patience=5),
                  DEVICE, log=None)
probe_curve = pd.DataFrame(probe.history)[["epoch", "train_loss", "val_score"]]
probe_curve["val_score"] = pd.to_numeric(probe_curve["val_score"], errors="coerce")
display(probe_curve)
falling = probe_curve["train_loss"].iloc[-1] < probe_curve["train_loss"].iloc[0]
rising = probe_curve["val_score"].iloc[-1] > probe_curve["val_score"].iloc[0]
print(f"train loss falling: {falling} | validation score rising: {rising} | "
      f"{probe.seconds / 5:.1f} s per epoch")

,epoch,train_loss,val_score
0,0,6.247321,0.377599
1,1,5.203534,0.393609
2,2,4.941076,0.418863
3,3,4.739991,0.442122
4,4,4.591699,0.461122


train loss falling: True | validation score rising: True | 4.2 s per epoch


### The sweep

This is the long cell. It is resumable twice over: a finished run is read back from its JSON
rather than retrained, and an unfinished one resumes from a checkpoint written every ten
epochs. Interrupting it is safe, and so is rerunning it.

On a laptop, stop the machine sleeping first. In a terminal, from anywhere:

```
caffeinate -i -w $(pgrep -n -f ipykernel)
```

In [7]:
# The long cell. Safe to interrupt and rerun: finished runs are read back, unfinished ones
# resume from their last checkpoint.
started = time.perf_counter()
for variant, overrides in RUNS:
    config = train.TrainConfig(batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
                               channels=CHANNELS, **overrides)
    train.run(STREAMS, variant, config, DEVICE, RESULTS_DIR)
print(f"\nsweep finished in {(time.perf_counter() - started) / 3600:.2f} hours")

  fungal_expression__unimodal-prot__lambda0.0__ch100__appendix__seed0: already run, test 0.7063
  fungal_expression__unimodal-dna__lambda0.0__ch100__appendix__seed0: already run, test 0.8140
  fungal_expression__unimodal-rna__lambda0.0__ch100__appendix__seed0: already run, test 0.7224
  fungal_expression__concat__lambda0.0__ch100__appendix__seed0: already run, test 0.7819
  fungal_expression__mil__lambda0.01__ch100__appendix__seed0: already run, test 0.7869
  fungal_expression__mil__lambda0.5__ch100__appendix__seed0: already run, test 0.7939
  fungal_expression__mil__lambda1.0__ch100__appendix__seed0: already run, test 0.7662
  fungal_expression__mil__lambda0.0__ch100__appendix__seed0: already run, test 0.7854
  fungal_expression__cross__lambda0.0__ch100__appendix__seed0: already run, test 0.8281

sweep finished in 0.00 hours


## Did it reproduce?

Four questions, and they are not the same question. Absolute agreement is the weakest of
them, because the fusion gain on this benchmark is 0.020 and a build can land every variant
within a few hundredths while reversing the ordering the paper's claim rests on.

1. Does each variant land near its published value?
2. Do the three single-modality baselines come out in the published order? On fungal
   expression that order is DNA ahead of RNA ahead of protein, which is unusual among the
   five benchmarks and so a real test rather than a coincidence.
3. Does every fusion head beat the best single encoder? This is the paper's central claim.
4. Is the best of the three fusion heads the one the paper reports as best here? On fungal
   expression that is the gated head; on E. coli proteins it is cross-attention, so the
   check reads the answer out of Table 1 rather than assuming it.

Each is stated below as something that is true or false, not as a number to interpret.


In [8]:
# Read back whatever has finished, so this works on a partial sweep too.
PUBLISHED_OF = {
    "unimodal:dna":  ("Nucleotide Transformer", TABLE1.loc["Nucleotide Transformer", BENCHMARK]),
    "unimodal:rna":  ("RNA-FM",                 TABLE1.loc["RNA-FM", BENCHMARK]),
    "unimodal:prot": ("ESM-2 (8M)",             TABLE1.loc["ESM-2 (8M)", BENCHMARK]),
    "concat":        ("Concatenation",          TABLE1.loc["Concatenation", BENCHMARK]),
    "mil":           ("MIL + Entropy",          TABLE1.loc["MIL + Entropy", BENCHMARK]),
    "mil (lam 0)":   ("MIL without entropy",    TABLE4.loc["MIL without entropy", BENCHMARK]),
    "cross":         ("Cross Attention",        TABLE1.loc["Cross Attention", BENCHMARK]),
}

# Only the sweep's own setting: the section after this one runs the other convolution into
# the same folder, and pooling the two would quietly average two different experiments.
finished = []
for path in sorted(RESULTS_DIR.glob("*.json")):
    r = train.Result.load(path)
    if r.config.get("tconv", "appendix") != "appendix" or r.config["channels"] != CHANNELS:
        continue
    finished.append({"variant": r.variant, "lambda": r.config["entropy_lambda"],
                     "val": r.val_score, "test": r.test_score, "epochs": r.epochs,
                     "best epoch": r.best_epoch, "minutes": round(r.seconds / 60, 1),
                     "alpha": r.alpha})
finished = pd.DataFrame(finished)
if finished.empty:
    raise RuntimeError("no finished runs yet: run the sweep cell above")

# The paper picks the gated head's lambda on validation, so we do too; lambda 0 is the ablation.
gated = finished[(finished["variant"] == "mil") & (finished["lambda"] > 0)]
chosen_lambda = None if gated.empty else gated.loc[gated["val"].idxmax(), "lambda"]

rows = []
for _, r in finished.iterrows():
    if r["variant"] == "mil" and r["lambda"] == 0:
        key = "mil (lam 0)"
    elif r["variant"] == "mil" and r["lambda"] != chosen_lambda:
        continue                                   # a sweep member that validation did not pick
    else:
        key = r["variant"]
    name, published = PUBLISHED_OF[key]
    rows.append({"variant": key, "paper row": name, "ours": round(r["test"], 3),
                 "published": published, "difference": round(r["test"] - published, 3),
                 "epochs": r["epochs"], "minutes": r["minutes"]})
comparison = pd.DataFrame(rows).set_index("variant")
print(f"gated head: lambda {chosen_lambda} chosen on validation "
      f"from {sorted(gated['lambda'])}" if chosen_lambda is not None else "gated head: not run")
display(comparison)
display(finished[["variant", "lambda", "val", "test", "epochs", "minutes"]])

gated head: lambda 0.5 chosen on validation from [0.01, 0.5, 1.0]


,paper row,ours,published,difference,epochs,minutes
variant,,,,,,
concat,Concatenation,0.782,0.805,-0.023,33,4.2
cross,Cross Attention,0.828,0.812,0.016,34,18.6
mil (lam 0),MIL without entropy,0.785,0.826,-0.041,35,4.0
mil,MIL + Entropy,0.794,0.824,-0.030,35,4.7
unimodal:dna,Nucleotide Transformer,0.814,0.804,0.010,85,5.0
unimodal:prot,ESM-2 (8M),0.706,0.695,0.011,86,4.8
unimodal:rna,RNA-FM,0.722,0.767,-0.045,27,2.6


,variant,lambda,val,test,epochs,minutes
0,concat,0.00,0.696243,0.781942,33,4.2
1,cross,0.00,0.734546,0.828086,34,18.6
2,mil,0.01,0.701550,0.786872,33,4.6
3,mil,0.00,0.710728,0.785416,35,4.0
4,mil,0.50,0.709015,0.793865,35,4.7
5,mil,1.00,0.662853,0.766228,106,12.1
6,unimodal:dna,0.00,0.703497,0.813991,85,5.0
7,unimodal:prot,0.00,0.663796,0.706285,86,4.8
8,unimodal:rna,0.00,0.638369,0.722432,27,2.6


In [9]:
# The four questions, each as a statement that is true or false rather than a number to read.
# What counts as the published answer is read out of Table 1, so switching BENCHMARK above
# switches the expectations with it.
ours = comparison["ours"].to_dict()
SINGLE_ROWS = {"unimodal:dna": "Nucleotide Transformer", "unimodal:rna": "RNA-FM",
               "unimodal:prot": "ESM-2 (8M)"}
FUSION_ROWS = {"concat": "Concatenation", "mil": "MIL + Entropy", "cross": "Cross Attention"}
verdict = []


def claim(text, known, passed, detail):
    verdict.append({"claim": text, "result": "not yet" if not known else ("ok" if passed else "NO"),
                    "detail": detail})


def by_published(rows):
    return sorted(rows, key=lambda k: -TABLE1.loc[rows[k], BENCHMARK])


gaps = comparison["difference"].abs()
claim("every variant lands within 0.05 of its published value", True, bool((gaps <= 0.05).all()),
      f"largest gap {gaps.max():.3f} ({gaps.idxmax()})")

known = all(k in ours for k in SINGLE_ROWS)
published_order = by_published(SINGLE_ROWS)
our_order = sorted((k for k in SINGLE_ROWS if k in ours), key=lambda k: -ours[k])
claim("the three single encoders come out in the published order", known,
      known and our_order == published_order,
      "published " + " > ".join(published_order) + "; ours " +
      " > ".join(f"{k} {ours[k]:.3f}" for k in our_order))

fusions = [k for k in FUSION_ROWS if k in ours]
singles = [k for k in SINGLE_ROWS if k in ours]
known = len(fusions) == 3 and bool(singles)
best_single = max((ours[k] for k in singles), default=float("nan"))
claim("every fusion head beats the best single encoder", known,
      known and all(ours[f] >= best_single for f in fusions),
      f"best single {best_single:.3f}; " + ", ".join(f"{f} {ours[f]:.3f}" for f in fusions))

published_best = by_published(FUSION_ROWS)[0]
known = len(fusions) == 3
claim(f"the best fusion head is the paper's best here ({published_best})", known,
      known and max(fusions, key=lambda k: ours[k]) == published_best,
      ", ".join(f"{f} {ours[f]:.3f}" for f in fusions))

if "mil" in ours and "mil (lam 0)" in ours:
    our_gain = ours["mil"] - ours["mil (lam 0)"]
    paper_gain = (TABLE1.loc["MIL + Entropy", BENCHMARK]
                  - TABLE4.loc["MIL without entropy", BENCHMARK])
    claim("entropy moves the gated head the way the paper's own ablation does", True,
          (our_gain > 0) == (paper_gain > 0),
          f"ours {our_gain:+.3f}, paper {paper_gain:+.3f}; note the paper's text says entropy "
          f"improves performance while its Table 4 has it {paper_gain:+.3f} on this benchmark")

pd.DataFrame(verdict).set_index("claim")


,result,detail
claim,,
every variant lands within 0.05 of its published value,ok,largest gap 0.045 (unimodal:rna)
the three single encoders come out in the published order,ok,published unimodal:dna > unimodal:rna > unimod...
every fusion head beats the best single encoder,NO,"best single 0.814; concat 0.782, mil 0.794, cr..."
the best fusion head is the paper's best here (mil),NO,"concat 0.782, mil 0.794, cross 0.828"
entropy moves the gated head the way the paper's own ablation does,NO,"ours +0.009, paper -0.002; note the paper's te..."


## The transposed convolution: the paper's two settings are not equivalent

The DNA upsampling is specified twice. Equation 1 gives kernel 2, stride 2. Appendix List 1
and List 2 give kernel 3, stride 2, padding 2. The sweep above used the appendix setting,
because the appendix is what the paper calls the detailed architecture.

They are not two spellings of the same thing. A transposed convolution with kernel 3 and
stride 2 produces `2n + 1` positions before padding is removed, and padding 2 removes two
from each end, so what comes out is the *middle* of the natural upsampling: shifted by two
positions, and three short of the codon frame. Kernel 2 with stride 2 produces exactly `2n`
positions with nothing trimmed.

Two codons is six nucleotides. The cell below measures it rather than arguing it: the
convolution is set to copy its input, a single DNA token is sent through, and the codons it
lands on are printed.

In [10]:
# Where does one DNA 6-mer token land on the codon frame? With the convolution set to copy
# its input, token t should light up codons 2t and 2t+1, because a 6-mer covers two codons.
for setting in codon_grid.TCONV_SETTINGS:
    probe_grid = codon_grid.CodonGrid(1, tconv=setting).eval()
    with torch.no_grad():
        probe_grid.dna_upsample.weight.fill_(1.0)
        probe_grid.dna_upsample.bias.fill_(0.0)
    tokens, frame = 20, 40                                 # 120 nt: 20 six-mers, 40 codons
    produced = (tokens - 1) * probe_grid.stride - 2 * probe_grid.padding + probe_grid.kernel
    print(f"{setting:<9} kernel {probe_grid.kernel} stride {probe_grid.stride} "
          f"padding {probe_grid.padding} -> {produced} positions for a {frame}-codon frame")
    for t in (0, 1, 5, 19):
        one = torch.zeros(1, tokens, 1)
        one[0, t, 0] = 1.0
        with torch.no_grad():
            landed, _ = probe_grid(one, torch.zeros(1, 120, 1), torch.zeros(1, frame, 1),
                                   torch.tensor([frame]), torch.tensor([tokens]))
        where = (landed["dna"][0, :, 0] != 0).nonzero().flatten().tolist()
        expected = [2 * t, 2 * t + 1]
        print(f"    6-mer {t:>2} -> codons {str(where):<14}"
              + ("" if where == expected else f"should be {expected}"))
    print()

appendix  kernel 3 stride 2 padding 2 -> 37 positions for a 40-codon frame
    6-mer  0 -> codons [0]           should be [0, 1]
    6-mer  1 -> codons [0, 1, 2]     should be [2, 3]
    6-mer  5 -> codons [8, 9, 10]    should be [10, 11]
    6-mer 19 -> codons [36]          should be [38, 39]

equation  kernel 2 stride 2 padding 0 -> 40 positions for a 40-codon frame
    6-mer  0 -> codons [0, 1]        
    6-mer  1 -> codons [2, 3]        
    6-mer  5 -> codons [10, 11]      
    6-mer 19 -> codons [38, 39]      



### What that predicts, and the rerun that tests it

A two-codon shift damages a head that combines the streams position by position, and leaves
alone a head that does not:

| head | how it combines the streams | predicted effect of the shift |
| --- | --- | --- |
| concatenation | joins them at each codon | hurt |
| gated MIL | weighted sum at each codon | hurt |
| cross-attention | attends to the whole joint context | little or none |
| the unimodal baselines | never touch the grid | none |

That is the pattern the sweep produced, which is suggestive but not a test: the comparison
above was made against published numbers, and plenty of things could move a score by 0.03.

There is a second piece of evidence that is harder to explain another way. The gated head
reports what it did with each modality, and at the λ validation chose, **its DNA weight is
0.00** while DNA is the strongest single modality on this benchmark. A head that learns to
discard its best input is saying that input arrives damaged.

The real test is below: the four grid-using configurations rerun under Equation 1, nothing
else changed. The unimodal baselines are not rerun, because the grid never touches them. If
the shift is the cause, concatenation and the gated head should rise and cross-attention
should barely move. If they all move together, or none do, the cause is something else.

In [11]:
# The same sweep minus the unimodal baselines, under Equation 1's convolution. Roughly the
# cost of the first sweep less the three baselines; resumable in the same way.
EQUATION_RUNS = ([("concat", {})]
                 + [("mil", {"entropy_lambda": lam}) for lam in LAMBDAS]
                 + [("mil", {"entropy_lambda": 0.0}), ("cross", {})])
started = time.perf_counter()
for variant, overrides in EQUATION_RUNS:
    train.run(STREAMS, variant,
              train.TrainConfig(batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
                                channels=CHANNELS, tconv="equation", **overrides),
              DEVICE, RESULTS_DIR)
print(f"\nrerun finished in {(time.perf_counter() - started) / 3600:.2f} hours")

  fungal_expression__concat__lambda0.0__ch100__equation__seed0: already run, test 0.7968
  fungal_expression__mil__lambda0.01__ch100__equation__seed0: already run, test 0.8018
  fungal_expression__mil__lambda0.5__ch100__equation__seed0: already run, test 0.7943
  fungal_expression__mil__lambda1.0__ch100__equation__seed0: already run, test 0.7708
  fungal_expression__mil__lambda0.0__ch100__equation__seed0: already run, test 0.8022
  fungal_expression__cross__lambda0.0__ch100__equation__seed0: already run, test 0.8235

rerun finished in 0.00 hours


In [12]:
# Both settings beside the published values. The gated head's lambda is picked on validation
# separately within each setting, because that is what the paper's procedure would do.
def collect(tconv):
    runs = [train.Result.load(p) for p in sorted(RESULTS_DIR.glob("*.json"))]
    runs = [r for r in runs if r.config.get("tconv", "appendix") == tconv
            and r.config["channels"] == CHANNELS and r.config["seed"] == 0]
    swept = [r for r in runs if r.variant == "mil" and r.config["entropy_lambda"] > 0]
    picked = max(swept, key=lambda r: r.val_score).config["entropy_lambda"] if swept else None
    scores = {}
    for r in runs:
        lam = r.config["entropy_lambda"]
        if r.variant == "mil" and lam == 0:
            scores["mil (lam 0)"] = r.test_score
        elif r.variant == "mil" and lam != picked:
            continue
        else:
            scores[r.variant] = r.test_score
    return scores, picked


appendix, lambda_appendix = collect("appendix")
equation, lambda_equation = collect("equation")
grid_users = [k for k in ("concat", "mil", "mil (lam 0)", "cross") if k in appendix]
settings = pd.DataFrame({
    "published": {k: PUBLISHED_OF[k][1] for k in grid_users},
    "appendix (k3 s2 p2)": {k: appendix.get(k) for k in grid_users},
    "Equation 1 (k2 s2)": {k: equation.get(k) for k in grid_users},
})
settings["change"] = settings["Equation 1 (k2 s2)"] - settings["appendix (k3 s2 p2)"]
settings["Eq 1 vs published"] = settings["Equation 1 (k2 s2)"] - settings["published"]
display(settings.round(3))
print(f"lambda chosen on validation: appendix {lambda_appendix}, Equation 1 {lambda_equation}")

# And what the gated head now does with DNA, which is the other half of the evidence.
for tconv in ("appendix", "equation"):
    for p in sorted(RESULTS_DIR.glob("*.json")):
        r = train.Result.load(p)
        if r.variant == "mil" and r.alpha and r.config.get("tconv", "appendix") == tconv \
                and r.config["entropy_lambda"] == (lambda_appendix if tconv == "appendix"
                                                   else lambda_equation):
            print(f"{tconv:<9} gated attention weights: "
                  + "  ".join(f"{k} {v:.2f}" for k, v in r.alpha.items()))

,published,appendix (k3 s2 p2),Equation 1 (k2 s2),change,Eq 1 vs published
concat,0.805,0.782,0.797,0.015,-0.008
mil,0.824,0.794,0.802,0.008,-0.022
mil (lam 0),0.826,0.785,0.802,0.017,-0.024
cross,0.812,0.828,0.824,-0.005,0.012


lambda chosen on validation: appendix 0.5, Equation 1 0.01
appendix  gated attention weights: dna 0.00  rna 0.48  prot 0.52
equation  gated attention weights: dna 0.20  rna 0.20  prot 0.60


## How much of this is measurement noise?

Equation 1 moved the two position-wise heads up and left cross-attention alone, which is the
signature the misalignment predicts. But before reading anything into what is left, the
scores need an error bar, and so far every number here is a single run scored once on a
single split.

The fungal test split is **430 rows**. A Spearman correlation measured on 430 rows carries a
95% interval of roughly ±0.03, by Fisher's transformation with the standard correction for
Spearman. The cell below works it out for the values in hand. The differences being argued
over, ours against published and fusion against the best single encoder, are between 0.008
and 0.024. They are all smaller than the interval on any one of the numbers.

That cuts in two directions, and the second is the interesting one. It means this
reproduction may be closer than the table suggests. It also means **the paper's own headline
on this benchmark, fusion at 0.824 against the best single encoder at 0.804, is a gap of
0.020 reported from one split with no error bar**, and 0.020 is inside the same interval.

In [13]:
# How precise is a Spearman correlation on this test split at all? Fisher's z with the
# Fieller correction for Spearman: the standard error on the z scale is 1.03 / sqrt(n - 3).
test_rows = len(STREAMS.split_rows("test"))
z_error = 1.03 / np.sqrt(test_rows - 3)
interval = pd.DataFrame([
    {"score": s,
     "95% low": np.tanh(np.arctanh(s) - 1.96 * z_error),
     "95% high": np.tanh(np.arctanh(s) + 1.96 * z_error)}
    for s in sorted(set(list(settings["published"]) + list(settings["Equation 1 (k2 s2)"].dropna())
                        + [appendix[k] for k in appendix if k.startswith("unimodal")]))])
interval["half-width"] = (interval["95% high"] - interval["95% low"]) / 2
print(f"test split: {test_rows} rows")
display(interval.round(3).set_index("score"))

test split: 430 rows


,95% low,95% high,half-width
score,,,
0.706,0.654,0.752,0.049
0.722,0.672,0.766,0.047
0.797,0.758,0.830,0.036
0.802,0.764,0.834,0.035
0.802,0.765,0.834,0.035
0.805,0.768,0.837,0.034
0.812,0.776,0.843,0.033
0.814,0.778,0.844,0.033
0.824,0.789,0.853,0.032


### What an error bar would have to come from

Two different sources of variation, and they need different treatment.

**The split.** Scoring on 430 particular rows is one draw. The right way to compare two
models here is not to ask whether their separate intervals overlap, because they are scored
on *the same rows*: that is a paired comparison, and two models that agree closely row by row
can be reliably separated by far less than either interval suggests. Resampling the rows and
looking at the distribution of the **difference** is the test. That needs per-row predictions,
which `train.py` now saves with every result; the runs already on disk predate it, so the
four configurations below are retrained once to produce them.

**The seed.** One initialization, one shuffle. The paper does not say how many seeds it used
or whether its numbers are means. Three seeds will not give a tight interval but will show
whether the spread is nearer 0.005 or 0.03, which is enough to know what the comparison can
support.

The sweep below is the four configurations the central claim actually involves: the best
single encoder, and the three fusion heads, all under Equation 1. Twelve runs, roughly an
hour and a half, most of it cross-attention. The gated head is held at the λ validation
chose under Equation 1 rather than reselecting per seed, which is a simplification worth
recording.

In [14]:
# Four configurations, three seeds, Equation 1. require_predictions retrains the seed-0 runs
# once, because they were written before per-row predictions were saved.
SEEDS = (0, 1, 2)
CLAIM_RUNS = [("unimodal:dna", {}), ("concat", {}),
              ("mil", {"entropy_lambda": lambda_equation}), ("cross", {})]
started = time.perf_counter()
for seed in SEEDS:
    print(f"\n--- seed {seed} ---")
    for variant, overrides in CLAIM_RUNS:
        train.run(STREAMS, variant,
                  train.TrainConfig(batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
                                    channels=CHANNELS, tconv="equation", seed=seed,
                                    **overrides),
                  DEVICE, RESULTS_DIR, require_predictions=True)
print(f"\nseed sweep finished in {(time.perf_counter() - started) / 3600:.2f} hours")


--- seed 0 ---
  fungal_expression__unimodal-dna__lambda0.0__ch100__equation__seed0: training
    epoch   0  train loss 5.4896  val 0.5162  best 0.5162 @ 0
    epoch  10  train loss 3.3860  val 0.6486  best 0.6486 @ 10
    epoch  20  train loss 2.4631  val 0.6702  best 0.6702 @ 20
    epoch  30  train loss 1.7329  val 0.6829  best 0.6829 @ 30
    epoch  40  train loss 1.0838  val 0.6916  best 0.6916 @ 40
    epoch  50  train loss 0.6398  val 0.6971  best 0.6971 @ 50
    epoch  60  train loss 0.3452  val 0.6988  best 0.6988 @ 60
    epoch  70  train loss 0.1991  val 0.6989  best 0.6994 @ 64
    epoch  80  train loss 0.1990  val 0.6993  best 0.6994 @ 64
  fungal_expression__unimodal-dna__lambda0.0__ch100__equation__seed0: test 0.8125 (val 0.6994 at epoch 64, 4.1 min)
  fungal_expression__concat__lambda0.0__ch100__equation__seed0: retraining, the stored result has no per-row predictions
  fungal_expression__concat__lambda0.0__ch100__equation__seed0: training
    epoch   0  train loss 5.5

In [15]:
# Spread across seeds, then the paired test of the paper's central claim.
claim = {}
for variant, overrides in CLAIM_RUNS:
    scores, runs = [], []
    for seed in SEEDS:
        tag = (f"{BENCHMARK}__{variant.replace(':', '-')}"
               f"__lambda{overrides.get('entropy_lambda', 0.0)}__ch{CHANNELS}"
               f"__equation__seed{seed}.json")
        if (RESULTS_DIR / tag).exists():
            r = train.Result.load(RESULTS_DIR / tag)
            scores.append(r.test_score)
            runs.append(r)
    if scores:
        claim[variant] = {"runs": runs, "scores": scores}

spread = pd.DataFrame([
    {"variant": v, "seeds": len(d["scores"]), "mean": np.mean(d["scores"]),
     "min": min(d["scores"]), "max": max(d["scores"]),
     "spread": max(d["scores"]) - min(d["scores"]),
     "published": PUBLISHED_OF[v][1]}
    for v, d in claim.items()]).set_index("variant")
spread["mean vs published"] = spread["mean"] - spread["published"]
display(spread.round(3))

# The claim: does each fusion head beat the best single encoder, on the same rows?
# Averaged over seeds rather than one seed at a time. All three seeds are scored on the same
# 430 rows, so the rows are resampled once per draw and the seed-to-seed average is taken
# inside the draw; that removes initialization noise from the interval instead of pretending
# the seeds are independent samples.
baseline = claim.get("unimodal:dna")
if baseline and len(baseline["runs"]) == len(SEEDS):
    truth = np.asarray(baseline["runs"][0].test_truth)
    rows = np.arange(len(truth))
    predictions = {v: [np.asarray(r.test_prediction) for r in d["runs"]]
                   for v, d in claim.items()}
    rng = np.random.default_rng(0)
    draws = [rng.integers(0, len(truth), len(truth)) for _ in range(4000)]

    def averaged(variant, pick):
        return np.mean([train.score_rows(STREAMS.task, truth[pick], p[pick])
                        for p in predictions[variant]])

    def compare(label, a, b):
        spread = np.array([averaged(a, pick) - averaged(b, pick) for pick in draws])
        low, high = np.percentile(spread, [2.5, 97.5])
        return {"comparison": label, "difference": averaged(a, rows) - averaged(b, rows),
                "95% low": low, "95% high": high, "P(>0)": (spread > 0).mean(),
                "resolved": "yes" if low > 0 or high < 0 else "no"}

    tests = [compare(f"{h} minus best single encoder", h, "unimodal:dna")
             for h in ("concat", "mil", "cross") if h in claim]
    tests += [compare("cross minus mil", "cross", "mil"),
              compare("cross minus concat", "cross", "concat"),
              compare("mil minus concat", "mil", "concat")]
    display(pd.DataFrame(tests).set_index("comparison").round(3))
    print("'resolved' is yes only where the interval excludes zero. Note which comparisons\n"
          "resolve: two fusion heads make correlated errors, so their difference is measured\n"
          "far more precisely than the difference between a fusion head and a unimodal model.")

,seeds,mean,min,max,spread,published,mean vs published
variant,,,,,,,
unimodal:dna,3,0.807,0.801,0.813,0.011,0.804,0.003
concat,3,0.791,0.789,0.793,0.004,0.805,-0.014
mil,3,0.801,0.797,0.804,0.007,0.824,-0.023
cross,3,0.823,0.814,0.831,0.017,0.812,0.011


minus best single  95% low  95% high  P(fusion wins)
fusion head seed                                                      
concat      0                -0.020   -0.054     0.015           0.136
            1                -0.008   -0.043     0.027           0.336
            2                -0.017   -0.051     0.016           0.158
mil         0                -0.011   -0.049     0.028           0.298
            1                 0.003   -0.034     0.043           0.587
            2                -0.009   -0.049     0.030           0.338
cross       0                 0.011   -0.024     0.047           0.732
            1                 0.013   -0.018     0.048           0.818
            2                 0.025   -0.010     0.060           0.930

A head beats the best single encoder only where the interval excludes zero.


## Deviations recorded for this step

The ones this notebook adds, on top of the two recorded in step 1 and step 2 (dropping
sequences over 999 nucleotides, and translating every codon with stops written as X).

| # | deviation | why | effect if wrong |
| --- | --- | --- | --- |
| 1 | TextCNN width 100 for every variant | List 1 says 1,280 for the gated head and List 2 says 100 for cross-attention, while the text calls the head fixed; one width is what "fair comparison" means | the gated head is under-capacity relative to List 1, which the extra run below tests |
| 2 | Transposed convolution kernel 3, stride 2, padding 2 for the main sweep | Appendix List 1 and List 2, against Equation 1's kernel 2, stride 2 | measured above: this setting puts the DNA stream two codons out of register and three short of the frame, while Equation 1 lands exactly. Both are run; whichever wins is what Stage 4 should inherit |
| 3 | DNA projected to 600, not to a width chosen because 4,107 is large | Table 3's 4,107 is the checkpoint's vocabulary size, not its hidden size, which is 512 | none expected; the projection is kept because it is in the architecture |
| 4 | The DNA stream is trimmed to each row's own convolution output length | otherwise a short row batched with a long one gets real outputs where it should have padding | without it, a prediction depends on batch composition, which is unreproducible |
| 5 | The gated head's per-modality mean is taken over real codons only | same reason | same |
| 6 | Early stopping and the scheduler watch the validation score | the paper names both mechanisms and their patience but not what they monitor | a different monitor would stop at a different epoch |
| 7 | An undefined validation score counts as no improvement | Spearman is NaN when nothing varies, which the first epochs can produce | without it a run ends with no weights to restore |

Still open from step 1 and unchanged here: **Ab1 is missing**, and it is the benchmark where
token-level cross-attention gains most, so the one claim this reproduction cannot test is the
one about token-level interaction. The paper sources it to Yazdani et al. (2024), whose first
author is at UCF. Worth asking Mina to make the introduction.

## The optional extra run: does head width explain any gap?

Only worth running if the gated head came in below 0.824. List 1 specifies 1,280 Conv1D
output channels for it, nearly thirteen times the width used above, and that is the single
largest unforced difference between this build and the paper's. It costs roughly five times
a normal gated run.

In [16]:
# Set to True to run the gated head at List 1's width, at the lambda validation chose.
RUN_WIDE_HEAD = False

if RUN_WIDE_HEAD:
    wide = train.run(STREAMS, "mil",
                     train.TrainConfig(batch_size=BATCH_SIZE, max_epochs=MAX_EPOCHS,
                                       channels=1280, entropy_lambda=lambda_equation,
                                       tconv="equation"),
                     DEVICE, RESULTS_DIR, require_predictions=True)
    print(f"gated head at 1,280 channels: {wide.test_score:.3f} "
          f"(at 100 channels: {equation['mil']:.3f}, published: {PUBLISHED_OF['mil'][1]})")
else:
    print("not run; set RUN_WIDE_HEAD = True above to run it")

not run; set RUN_WIDE_HEAD = True above to run it


---

## What step 4 inherits

If the four claims above hold, the model in `../shared_code/fusion/` is a checked
reimplementation of BioLangFusion and every Stage 4 experiment is measured against it rather
than against an assumption. Stage 4 then adds, in the order Section 12 of the write-up gives:

- the same six variants on the other three benchmarks, with mRNA stability reported twice,
  once on the published split and once on a split without the 68 percent sequence overlap
- the alignment ablation, which is the codon grid against pooled vectors
- the encoder swaps, which is the point of Stage 1's twelve frozen encoders
- Coupled Mamba and cross-attention as a fourth head, in January

The one thing to settle before the swaps: a stream whose tokens do not divide evenly into the
codon grid. Nucleotide Transformer v2 gives one token per six nucleotides, two codons, so an
odd codon count leaves half a token, which the transposed convolution's factor of two cannot
produce. The fit-to-frame step handles it by padding, which is the paper's own answer, but a
6-mer encoder swapped in for a 1-mer one will meet it on every odd-length sequence rather than
occasionally. Worth deciding deliberately rather than inheriting.